In [1]:
print("ok")

ok


In [1]:
from dotenv import load_dotenv

In [2]:
from langchain_groq import ChatGroq

In [3]:
llm= ChatGroq(model = "groq/compound-mini")

In [4]:
llm.invoke("How are you?")

AIMessage(content="I'm doing well, thank you! How can I help you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 443, 'total_tokens': 533, 'completion_time': 0.205856, 'completion_tokens_details': None, 'prompt_time': 0.021212, 'prompt_tokens_details': None, 'queue_time': 0.343355, 'total_time': 0.227067}, 'model_name': 'groq/compound-mini', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a009ed-baa7-7670-9c34-3cb8290a6a75-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 443, 'output_tokens': 90, 'total_tokens': 533})

In [5]:
from langchain.tools import tool

Multiple ways of creating a tool from llm

Method 1: Using the decorateor @tool

In [6]:
@tool
def multiply(a: int, b: int) -> int:
    """
    Multiply two integers.

    Args:
        a (int): The first integer.
        b (int): The second integer.

    Returns:
        int: The product of a and b.
    """
    return a * b

In [7]:
multiply

StructuredTool(name='multiply', description='Multiply two integers.\n\nArgs:\n    a (int): The first integer.\n    b (int): The second integer.\n\nReturns:\n    int: The product of a and b.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x00000244DEA58C20>)

Method 2: Using Structuredtool

In [12]:
from langchain_core.tools import StructuredTool

Creating pydantic class for validation

In [23]:
from typing import ClassVar,Type
from pydantic import BaseModel, Field

In [16]:
class WeatherInput(BaseModel):
    city: str

In [9]:
def get_weather(city: str) -> str:
    """
    Get the weather for a given city.

    Args:
        city (str): The name of the city.

    Returns:
        str: A string describing the weather in the city.
    """
    return f"The weather in {city} is sunny."

In [17]:
weather_tool = StructuredTool.from_function(
    func=get_weather,
    name="get_weather",
    description="Fetches real-time weather data for a city",
    args_schema=WeatherInput,  
)

In [24]:
class WeatherInput(BaseModel):#pydentic class
    city: str = Field(..., description="City name")
    units: str = Field("metric", description="metric or imperial")

class GetWeatherTool(StructuredTool):#structuredTool will be used as a tool
    name: ClassVar[str] = "get_weather"           
    description: ClassVar[str] = (
        "Fetches weather data for a city"
    )
    args_schema: ClassVar[Type[BaseModel]] = WeatherInput

    def _run(self, city: str, units: str = "metric") -> str:
        return get_weather(city, units)